# TV OFF — Kaggle GPU connectivity test
Private smoke test only. No model downloads, no publishing, no production changes.


In [ ]:
import json, os, platform, subprocess, sys, glob
print('TV OFF Kaggle smoke test started')
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('NVIDIA device nodes:', glob.glob('/dev/nvidia*'))
try:
    p = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=20)
    print('NVIDIA_SMI_EXIT:', p.returncode)
    print('NVIDIA_SMI_OUTPUT:', (p.stdout or p.stderr)[:4000])
except Exception as e:
    print('NVIDIA_SMI_ERROR:', type(e).__name__, str(e)[:300])
result = {'status':'ok','cuda':False,'nvidia_device_nodes':glob.glob('/dev/nvidia*')}
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('PyTorch CUDA build:', torch.version.cuda)
    result['cuda'] = bool(torch.cuda.is_available())
    if result['cuda']:
        result['gpu'] = torch.cuda.get_device_name(0)
        result['vram_gib'] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
except Exception as e:
    print('PyTorch probe error:', type(e).__name__, str(e)[:300])
with open('/kaggle/working/tv_off_gpu_smoke_result.json','w') as f:
    json.dump(result,f,indent=2)
print('RESULT_JSON:', json.dumps(result))
